# Phase 1: LiDAR bag → merged map → mesh → web viewer
Runs on **free Colab (CPU is enough; no GPU needed for Phase 1)**. Every stage saves to Google Drive and is **resumable**: if Colab disconnects, re-run the cells and finished stages are skipped.

**Before running:** push the `hilti3d` repo to your GitHub and set `REPO_URL` below (or upload the repo folder to Colab).

In [ ]:
#@title 1. Settings
REPO_URL   = "https://github.com/YOUR_USER/hilti3d.git"   #@param {type:"string"}
BAG_FILE   = "rosbags/exp21_outside_building.bag"          #@param {type:"string"}
WORK_DIR   = "/content/drive/MyDrive/hilti_work/exp21"      #@param {type:"string"}
T_END      = 60   #@param {type:"number"}   # seconds to process; 0 = whole bag (153 s)
DATA_DIR   = "/content/data"   # bag is downloaded here (Colab local disk, NOT Drive)

In [ ]:
#@title 2. Mount Drive + get code + install
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess, sys
if not os.path.exists('/content/hilti3d'):
    subprocess.run(['git','clone',REPO_URL,'/content/hilti3d'], check=True)
%cd /content/hilti3d
!pip install -q -r requirements.txt
import open3d, kiss_icp, rosbags
print('Python', sys.version.split()[0], '| open3d', open3d.__version__)

In [ ]:
#@title 3. Download the bag straight from Hugging Face (no token needed; ~12 GB, a few minutes)
from huggingface_hub import hf_hub_download
os.makedirs(DATA_DIR, exist_ok=True)
bag_path = hf_hub_download(repo_id="Hilti-Research/hilti-slam-challenge-2022", repo_type="dataset",
                           filename=BAG_FILE, local_dir=DATA_DIR)
print(bag_path, round(os.path.getsize(bag_path)/1e9,2), 'GB')

In [ ]:
#@title 4. Run the pipeline (extract -> odometry -> map -> mesh -> export)
t_end_arg = f"--set t_end={T_END}" if T_END else ""
!python -m hilti3d.pipeline --config configs/exp21.yaml --bag "{bag_path}" --work "{WORK_DIR}" {t_end_arg}

If a stage fails, the log tells you which one; fix/adjust and re-run the cell above (finished stages are skipped). Use `--force` to redo everything, or `--stages mesh --force` to redo only the mesh. Any config value can be overridden with `--set key=value` (see `hilti3d/config.py`).

In [ ]:
#@title 5. Quick look: trajectory + map (top-down)
import numpy as np, json, matplotlib.pyplot as plt, open3d as o3d
P = np.load(f"{WORK_DIR}/poses.npy")
pcd = o3d.io.read_point_cloud(f"{WORK_DIR}/map.ply")
pts = np.asarray(pcd.points); sel = np.random.default_rng(0).choice(len(pts), min(300000, len(pts)), replace=False)
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
ax[0].scatter(pts[sel,0], pts[sel,1], s=0.2, c=pts[sel,2], cmap='viridis'); ax[0].plot(P[:,0,3], P[:,1,3], 'r-', lw=1)
ax[0].set_aspect('equal'); ax[0].set_title('Map (top-down, colour = height) + sensor path')
ax[1].scatter(pts[sel,0], pts[sel,2], s=0.2, c='k'); ax[1].set_aspect('equal'); ax[1].set_title('Side view (x vs z)')
plt.show()
for n in ['map_stats.json','mesh_stats.json']:
    print(n, json.load(open(f"{WORK_DIR}/{n}")))

In [ ]:
#@title 6. Download the viewer bundle (open with:  python -m http.server 8000  inside the folder)
import shutil
shutil.make_archive('/content/viewer', 'zip', f"{WORK_DIR}/viewer")
from google.colab import files
files.download('/content/viewer.zip')